In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr
import re
import matplotlib.pyplot as plt

# ==================== 配置 ====================
class Config:
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_PATH = './data/f1_model_balanced.pth'
    DATA_DIR = './data/'
    
    CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
    NUM_COLS = ['year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
                'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
                'Driver_Prev_Season_FL_Count', 'Is_Home_Race', 'Recent_3_Races_Avg_Pos', 
                'Performance_Trend', 'Consistency_Score']
    TARGET_COL = 'is_winner'
    
    EMB_DIM = 32
    HIDDEN_DIM = 128
    DROPOUT_RATE = 0.4
    BATCH_SIZE = 256
    LEARNING_RATE = 5e-4
    N_EPOCHS = 50
    PATIENCE = 10

def set_seeds():
    """設定隨機種子確保結果可重現"""
    np.random.seed(Config.SEED)
    torch.manual_seed(Config.SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(Config.SEED)

# ==================== 數據處理類 ====================
class DataProcessor:
    """統一的數據處理工具類"""
    
    @staticmethod
    def clean_string(text):
        """清理文字，移除多餘空格"""
        return re.sub(r'\s+', ' ', text).strip() if isinstance(text, str) else text
    
    @staticmethod
    def get_country_from_gp(gp_name):
        """從GP名稱獲取國家代碼用於主場優勢特徵"""
        gp_map = {
            'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
            'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
            'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
            'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
            'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
            'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
        }
        if not isinstance(gp_name, str):
            return None
        for key, country in gp_map.items():
            if key in gp_name:
                return country
        return None
    
    @staticmethod
    def load_and_clean_data():
        """載入並清理所有CSV數據"""
        d = Config.DATA_DIR
        
        # 讀取數據
        winners = pd.read_csv(d + 'winners.csv', encoding='utf-8')
        drivers = pd.read_csv(d + 'drivers_updated.csv', encoding='utf-8')
        teams = pd.read_csv(d + 'teams_updated.csv', encoding='utf-8')
        fastest_laps = pd.read_csv(d + 'fastest_laps_updated.csv', encoding='utf-8')
        
        # 清理文字欄位
        for df in [winners, drivers, teams, fastest_laps]:
            for col in df.select_dtypes(include='object'):
                df[col] = df[col].map(DataProcessor.clean_string)
        
        # 處理年份和位置
        winners['year'] = pd.to_datetime(winners['Date'], errors='coerce').dt.year.astype(int)
        drivers.rename(columns={'Car': 'Team'}, inplace=True)
        
        for df in [drivers, teams, fastest_laps]:
            df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
            if 'Pos' in df.columns:
                df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
        
        # 排除Indianapolis 500（非標準F1賽事）
        winners = winners[~winners['Grand Prix'].str.contains("Indianapolis 500", na=False)]
        fastest_laps = fastest_laps[~fastest_laps['Grand Prix'].str.contains("Indianapolis 500", na=False)]
        
        return winners, drivers, teams, fastest_laps

# ==================== 特徵工程 ====================
def create_enhanced_features(drivers, teams, fastest_laps):
    """創建增強的特徵集，包含lag特徵和動量特徵"""
    def_pos_drv, def_pos_team = 50, 20
    
    # === 基礎Lag特徵 ===
    drivers = drivers.sort_values(['Driver', 'year'])
    drivers['Prev_Year_Driver_PTS'] = drivers.groupby('Driver')['PTS'].shift(1).fillna(0)
    drivers['Prev_Year_Driver_Pos'] = drivers.groupby('Driver')['Pos'].shift(1).fillna(def_pos_drv)
    drivers['Driver_Experience_Years'] = drivers['year'] - drivers.groupby('Driver')['year'].transform('min')
    
    # 車隊lag特徵
    teams = teams.sort_values(['Team', 'year'])
    teams['Prev_Year_Team_PTS'] = teams.groupby('Team')['PTS'].shift(1).fillna(0)
    teams['Prev_Year_Team_Pos'] = teams.groupby('Team')['Pos'].shift(1).fillna(def_pos_team)
    teams['Team_Experience_Years'] = teams['year'] - teams.groupby('Team')['year'].transform('min')
    
    # 合併車隊特徵
    drivers = drivers.merge(teams[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']], 
                           on=['Team', 'year'], how='left').fillna(0)
    
    # === 最快圈速特徵 ===
    fl = fastest_laps.groupby(['year', 'Driver']).size().reset_index(name='FL_Count')
    fl['Driver_Prev_Season_FL_Count'] = fl.groupby('Driver')['FL_Count'].shift(1).fillna(0)
    drivers = drivers.merge(fl[['Driver', 'year', 'Driver_Prev_Season_FL_Count']], on=['Driver', 'year'], how='left').fillna(0)
    
    # === 關鍵新增：動量特徵 ===
    # 近期表現平均值
    drivers['Recent_3_Races_Avg_Pos'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).mean().reset_index(0, drop=True).fillna(def_pos_drv)
    
    # 表現趨勢（安全處理除零）
    drivers['PTS_prev'] = drivers.groupby('Driver')['PTS'].shift(1)
    drivers['Performance_Trend'] = 0.0
    mask = (drivers['PTS_prev'].notna()) & (drivers['PTS_prev'] > 0)
    drivers.loc[mask, 'Performance_Trend'] = ((drivers.loc[mask, 'PTS'] - drivers.loc[mask, 'PTS_prev']) / drivers.loc[mask, 'PTS_prev']).clip(-2.0, 2.0)
    drivers.drop('PTS_prev', axis=1, inplace=True)
    
    # 一致性評分
    drivers['Pos_Rolling_Std'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).std().reset_index(0, drop=True).fillna(0.0)
    drivers['Consistency_Score'] = 1.0 / (1.0 + drivers['Pos_Rolling_Std'])
    
    # 確保數值安全
    for col in ['Recent_3_Races_Avg_Pos', 'Performance_Trend', 'Consistency_Score']:
        drivers[col] = drivers[col].replace([float('inf'), float('-inf')], 0.0).fillna(0.0)
    
    return drivers

def build_modeling_dataset(winners, drivers_feat):
    """構建完整的建模數據集"""
    data = []
    drivers_by_year = {y: g for y, g in drivers_feat.groupby('year')}
    
    for _, race in winners.iterrows():
        year, gp, winner = race['year'], race['Grand Prix'], race['Winner']
        if year not in drivers_by_year:
            continue
            
        race_country = DataProcessor.get_country_from_gp(gp)
        for _, driver in drivers_by_year[year].iterrows():
            is_home = 1 if race_country and driver['Nationality'] == race_country else 0
            
            data.append({
                'year': year, 'Grand Prix': gp, 'Driver': driver['Driver'], 
                'Team': driver['Team'], 'Nationality': driver['Nationality'],
                'Prev_Year_Driver_PTS': driver['Prev_Year_Driver_PTS'],
                'Prev_Year_Driver_Pos': driver['Prev_Year_Driver_Pos'],
                'Driver_Experience_Years': driver['Driver_Experience_Years'],
                'Prev_Year_Team_PTS': driver['Prev_Year_Team_PTS'],
                'Prev_Year_Team_Pos': driver['Prev_Year_Team_Pos'],
                'Team_Experience_Years': driver['Team_Experience_Years'],
                'Driver_Prev_Season_FL_Count': driver['Driver_Prev_Season_FL_Count'],
                'Recent_3_Races_Avg_Pos': driver['Recent_3_Races_Avg_Pos'],
                'Performance_Trend': driver['Performance_Trend'],
                'Consistency_Score': driver['Consistency_Score'],
                'Is_Home_Race': is_home,
                'is_winner': int(driver['Driver'] == winner)
            })
    
    return pd.DataFrame(data)

def preprocess_features(train_df, test_df):
    """特徵預處理：編碼和標準化"""
    # 填補缺失值
    for col in Config.CAT_COLS:
        train_df[col] = train_df[col].fillna('Unknown')
        test_df[col] = test_df[col].fillna('Unknown')
    for col in Config.NUM_COLS:
        train_df[col] = train_df[col].fillna(0)
        test_df[col] = test_df[col].fillna(0)
    
    # 編碼分類特徵
    encoders, cat_dims = {}, {}
    for col in Config.CAT_COLS:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        test_df[col] = test_df[col].map(lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else len(le.classes_))
        encoders[col] = le
        cat_dims[col] = len(le.classes_) + 1
    
    # 標準化數值特徵
    scaler = StandardScaler()
    train_df[Config.NUM_COLS] = scaler.fit_transform(train_df[Config.NUM_COLS])
    test_df[Config.NUM_COLS] = scaler.transform(test_df[Config.NUM_COLS])
    
    return train_df, test_df, encoders, scaler, cat_dims

# ==================== 模型定義 ====================
class FocalLoss(nn.Module):
    """處理類別不平衡的Focal Loss"""
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, pred, target):
        pred_sigmoid = torch.sigmoid(pred)
        target = target.view(-1, 1)
        ce_loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        p_t = pred_sigmoid * target + (1 - pred_sigmoid) * (1 - target)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * ce_loss
        return focal_loss.mean()

class F1Dataset(Dataset):
    """F1數據集類"""
    def __init__(self, df):
        self.x_cat = df[Config.CAT_COLS].values
        self.x_num = df[Config.NUM_COLS].values
        self.y = df[Config.TARGET_COL].values
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (torch.tensor(self.x_cat[idx], dtype=torch.long),
                torch.tensor(self.x_num[idx], dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.float32))

class F1Model(nn.Module):
    """F1預測模型：embedding + DNN"""
    def __init__(self, cat_dims, num_feats):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, Config.EMB_DIM) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_feats)
        self.fc = nn.Sequential(
            nn.Linear(len(cat_dims) * Config.EMB_DIM + num_feats, Config.HIDDEN_DIM),
            nn.ReLU(), nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM, Config.HIDDEN_DIM // 2),
            nn.ReLU(), nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM // 2, 1)
        )
    
    def forward(self, x_cat, x_num):
        cat_emb = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], 1)
        num_norm = self.bn_num(x_num)
        combined = torch.cat([cat_emb, num_norm], 1)
        return self.fc(combined)

# ==================== 訓練和評估 ====================
def train_model(model, train_loader, test_loader):
    """訓練模型並保存最佳版本"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    criterion = FocalLoss()
    
    best_loss = float('inf')
    no_improve = 0
    train_losses, test_losses = [], []
    
    for epoch in range(Config.N_EPOCHS):
        # 訓練階段
        model.train()
        train_loss = 0
        for x_cat, x_num, y in train_loader:
            x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_cat, x_num), y.unsqueeze(1))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # 驗證階段
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for x_cat, x_num, y in test_loader:
                x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
                test_loss += criterion(model(x_cat, x_num), y.unsqueeze(1)).item()
        
        train_loss /= len(train_loader)
        test_loss /= len(test_loader)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        
        print(f"Epoch {epoch+1} | Train: {train_loss:.4f} | Test: {test_loss:.4f}")
        
        # 早停和模型保存
        if test_loss < best_loss:
            best_loss = test_loss
            torch.save(model.state_dict(), Config.MODEL_PATH)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= Config.PATIENCE:
                print("Early stopping")
                break
    
    # 保存訓練曲線
    plt.figure()
    plt.plot(train_losses, label='Train')
    plt.plot(test_losses, label='Test')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig('./data/loss_curve_balanced.png')
    plt.close()

def evaluate_model(model, test_loader):
    """評估模型性能"""
    if os.path.exists(Config.MODEL_PATH):
        model.load_state_dict(torch.load(Config.MODEL_PATH, map_location=Config.DEVICE))
    
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for x_cat, x_num, y in test_loader:
            x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
            probs = torch.sigmoid(model(x_cat, x_num))
            pred = (probs > 0.5).squeeze().int()
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy() if pred.ndim > 0 else [pred.item()])
    
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=['Not Winner', 'Winner'], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ==================== Gradio 預測介面 ====================
class F1Predictor:
    """F1賽事獲勝概率預測器"""
    def __init__(self):
        self.model = None
        self.encoders = {}
        self.scaler = None
        self.driver_data = {}
        self.years = []
        self.gps = []
    
    def setup(self, model, encoders, scaler, drivers, winners):
        """設置預測器組件"""
        self.model = model
        self.encoders = encoders
        self.scaler = scaler
        
        for year, group in drivers.groupby('year'):
            self.driver_data[year] = group.to_dict('records')
        
        self.years = sorted(drivers['year'].unique())
        self.gps = sorted(winners['Grand Prix'].unique())
    
    def predict_race_winner(self, year_input, gp_input):
        """預測指定賽事的獲勝概率排行"""
        try:
            year = int(year_input)
        except:
            return "錯誤：無效的年份輸入"
        
        if not gp_input or year not in self.driver_data:
            return "錯誤：無可用數據"
        
        self.model.eval()
        results = []
        
        for driver_info in self.driver_data[year]:
            # 處理分類特徵
            cat = []
            for col in Config.CAT_COLS:
                val = str(driver_info.get(col, 'Unknown'))
                if col == 'Grand Prix':
                    val = gp_input
                
                if val in self.encoders[col].classes_:
                    encoded = self.encoders[col].transform([val])[0]
                else:
                    encoded = len(self.encoders[col].classes_)
                cat.append(encoded)
            
            # 處理數值特徵
            num = []
            for col in Config.NUM_COLS:
                if col == 'Is_Home_Race':
                    race_country = DataProcessor.get_country_from_gp(gp_input)
                    driver_nationality = driver_info.get('Nationality', '')
                    val = 1.0 if race_country and driver_nationality == race_country else 0.0
                else:
                    val = float(driver_info.get(col, 0))
                num.append(val)
            
            # 預測
            x_num = torch.tensor(self.scaler.transform([num]), dtype=torch.float32).to(Config.DEVICE)
            x_cat = torch.tensor([cat], dtype=torch.long).to(Config.DEVICE)
            
            with torch.no_grad():
                prob = torch.sigmoid(self.model(x_cat, x_num)).cpu().item()
            
            results.append((driver_info.get('Driver', 'N/A'), driver_info.get('Team', 'N/A'), prob))
        
        # 返回前5名
        results.sort(key=lambda x: x[2], reverse=True)
        output = f"📊 {gp_input} {year} 獲勝概率預測：\n\n"
        for i, (driver, team, prob) in enumerate(results[:5], 1):
            output += f"{i}. {driver} ({team}): {prob:.2%}\n"
        
        return output.strip()
    
    def create_gradio_interface(self):
        """創建Gradio用戶界面"""
        with gr.Blocks(theme=gr.themes.Soft(), title="F1 Winner Predictor") as demo:
            gr.Markdown("# 🏎️ F1 大獎賽獲勝預測器")
            gr.Markdown("使用深度學習模型預測F1賽事獲勝概率（基於歷史數據和車手/車隊表現）")
            
            with gr.Row():
                year_dropdown = gr.Dropdown(
                    label="📅 選擇年份", 
                    choices=self.years, 
                    value=self.years[-1] if self.years else None
                )
                gp_dropdown = gr.Dropdown(
                    label="🏁 選擇大獎賽", 
                    choices=self.gps, 
                    value=self.gps[0] if self.gps else None
                )
            
            predict_button = gr.Button("🔮 預測獲勝者", variant="primary")
            output_textbox = gr.Textbox(
                label="🏆 預測結果 (前5名)", 
                lines=7, 
                interactive=False,
                placeholder="點擊預測按鈕查看結果..."
            )
            
            predict_button.click(
                self.predict_race_winner, 
                inputs=[year_dropdown, gp_dropdown], 
                outputs=[output_textbox]
            )
            
            with gr.Accordion("📈 訓練資訊", open=False):
                gr.Markdown("### 模型特徵包含：")
                gr.Markdown("- 車手/車隊歷史積分和排名\n- 經驗年數\n- 最快圈速記錄\n- **動量特徵**：近期表現趨勢\n- 主場優勢\n- 表現一致性評分")
                
                if os.path.exists("./data/loss_curve_balanced.png"):
                    gr.Image(value="./data/loss_curve_balanced.png", label="訓練損失曲線")
                else:
                    gr.Markdown("*訓練曲線圖片未找到*")
        
        return demo

# ==================== 主執行流程 ====================
def main():
    """主執行程序"""
    print("🚀 啟動 F1 獲勝預測系統...")
    
    # 初始化
    os.makedirs(Config.DATA_DIR, exist_ok=True)
    set_seeds()
    
    # 數據載入和特徵工程
    print("📊 載入數據並進行特徵工程...")
    winners, drivers, teams, fastest_laps = DataProcessor.load_and_clean_data()
    drivers_enhanced = create_enhanced_features(drivers, teams, fastest_laps)
    modeling_df = build_modeling_dataset(winners, drivers_enhanced)
    
    # 數據分割
    print("🔄 分割訓練和測試數據...")
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()
    
    if len(unique_race_ids) < 2:
        train_df = test_df = modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=Config.SEED)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    
    # 移除臨時欄位並預處理
    for df in [train_df, test_df]:
        if 'race_id' in df.columns:
            df.drop(columns=['race_id'], inplace=True)
    
    train_df, test_df, encoders, scaler, cat_dims = preprocess_features(train_df, test_df)
    
    # 創建數據載入器
    print("⚙️ 準備模型訓練...")
    train_set = F1Dataset(train_df)
    test_set = F1Dataset(test_df)
    
    weights = [1. / (train_df[Config.TARGET_COL].value_counts().get(t, 1) + 1e-6) for t in train_df[Config.TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    
    train_loader = DataLoader(train_set, batch_size=Config.BATCH_SIZE, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # 模型訓練
    cat_dims_ordered = [cat_dims[c] for c in Config.CAT_COLS]
    model = F1Model(cat_dims_ordered, len(Config.NUM_COLS)).to(Config.DEVICE)
    
    if not os.path.exists(Config.MODEL_PATH):
        print("🎯 開始模型訓練...")
        train_model(model, train_loader, test_loader)
    else:
        print("📦 載入已訓練的模型...")
    
    print("📈 模型評估結果：")
    evaluate_model(model, test_loader)
    
    # 啟動預測界面
    print("🌐 啟動 Gradio 預測界面...")
    predictor = F1Predictor()
    predictor.setup(model, encoders, scaler, drivers_enhanced, winners)
    demo = predictor.create_gradio_interface()
    
    print("✅ 系統就緒！正在啟動界面...")
    demo.launch(share=False)

if __name__ == '__main__':
    main()

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 啟動 F1 獲勝預測系統...
📊 載入數據並進行特徵工程...
🔄 分割訓練和測試數據...
⚙️ 準備模型訓練...
📦 載入已訓練的模型...
📈 模型評估結果：
Accuracy: 0.8803245436105477
              precision    recall  f1-score   support

  Not Winner       0.99      0.89      0.93      4710
      Winner       0.23      0.73      0.35       220

    accuracy                           0.88      4930
   macro avg       0.61      0.81      0.64      4930
weighted avg       0.95      0.88      0.91      4930

Confusion Matrix:
 [[4179  531]
 [  59  161]]
🌐 啟動 Gradio 預測界面...
✅ 系統就緒！正在啟動界面...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
